# Data Exploration: CARD and PATRIC Databases

This notebook explores the CARD and PATRIC/BV-BRC datasets for AMR prediction.

## Contents
1. Load and parse CARD database
2. Load and parse PATRIC database
3. Explore data statistics
4. Visualize data distributions
5. Build heterogeneous graph

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_preprocessing.card_parser import CARDParser
from src.data_preprocessing.patric_parser import PATRICParser
from src.data_preprocessing.graph_builder import HeterogeneousGraphBuilder

%matplotlib inline
sns.set_style('whitegrid')

## 1. Parse CARD Database

In [ ]:
# Initialize CARD parser
card_parser = CARDParser('../data/raw/card')

# Parse data
card_parser.parse_aro_ontology()
card_parser.parse_card_database()

# Extract entities
genes = card_parser.extract_genes()
drugs = card_parser.extract_drugs()
mechanisms = card_parser.extract_mechanisms()

# Get statistics
stats = card_parser.get_statistics()
print("CARD Statistics:")
for key, value in stats.items():
    print(f"  {key}: {value}")

## 2. Parse PATRIC Database

In [ ]:
# Initialize PATRIC parser
patric_parser = PATRICParser('../data/raw/patric')

# Parse data
patric_parser.parse_amr_data()
patric_parser.parse_genome_metadata()
patric_parser.parse_genome_features(sample_size=10000)  # Sample for quick testing

# Extract entities
species = patric_parser.extract_species()
amr_genes = patric_parser.extract_amr_genes()
phenotypes = patric_parser.extract_phenotypes()

# Get statistics
stats = patric_parser.get_statistics()
print("\nPATRIC Statistics:")
for key, value in stats.items():
    print(f"  {key}: {value}")

## 3. Build Heterogeneous Graph

In [ ]:
# Build graph
builder = HeterogeneousGraphBuilder(card_parser, patric_parser)
graph = builder.build()

# Get graph statistics
graph_stats = builder.get_statistics()
print("\nGraph Statistics:")
print(f"Node types: {graph_stats['num_node_types']}")
print(f"Edge types: {graph_stats['num_edge_types']}")
print("\nNodes:")
for node_type, count in graph_stats['num_nodes'].items():
    print(f"  {node_type}: {count}")
print("\nEdges:")
for edge_type, count in graph_stats['num_edges'].items():
    print(f"  {edge_type}: {count}")

## 4. Visualize Data

In [ ]:
# Plot node type distribution
fig, ax = plt.subplots(figsize=(10, 6))
node_counts = graph_stats['num_nodes']
ax.bar(node_counts.keys(), node_counts.values(), color='steelblue', alpha=0.8)
ax.set_ylabel('Count', fontsize=12)
ax.set_title('Node Type Distribution', fontsize=14, fontweight='bold')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 5. Save Processed Graph

In [ ]:
# Save graph
builder.save('../data/processed/graph.pkl')
print("Graph saved successfully!")